
# mt5-small (zh→en) — Baseline TF Training (Wuxia Domain)

**TFG Hugo Silvosa – Baseline NMT (MarianMT)**  
Este cuaderno entrena y evalúa un modelo **MarianMT** (`google/mt5-small`)  usando un dataset de **wuxia** (chino→inglés) ya preparado en formato `datasets` (HF).




## 1) Entorno de ejecución e instalación de dependencias

In [1]:

import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))



CUDA disponible: True
Número de GPUs: 1
Nombre de la GPU: NVIDIA GeForce RTX 3060



> **Requisitos del dataset**: directorio HF Datasets con *splits* `train`, `validation`, `test` y columnas `zh` (chino) y `en` (inglés):  
> `processed_data/wuxia_zh_en_clean/`

In [2]:
# Configuración de carpetas para entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent
BASE_DIR.mkdir(exist_ok=True)

# Estructura requerida por el usuario
for sub in ["training", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Estructura creada (si no existía):")
for p in ["trainign", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Nota: el dataset debe existir en: CORPUS/proccesed data/wuxia_zh_en_clean


Base: C:\Users\Usuario\Desktop\TFG\CORPUS
Estructura creada (si no existía):
 - C:\Users\Usuario\Desktop\TFG\CORPUS\trainign
 - C:\Users\Usuario\Desktop\TFG\CORPUS\models
 - C:\Users\Usuario\Desktop\TFG\CORPUS\proccesed data


## 2) Configuración

In [3]:
from dataclasses import dataclass

@dataclass
class Config:
    # Rutas (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"   
    output_dir: Path   = BASE_DIR / "models" / "mt5_small"             
    ckpt_dir: Path     = BASE_DIR / "checkpoints"
    training_dir: Path = BASE_DIR / "training"

    # Columnas del dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Modelo 
    model_ckpt: str = "google/mt5-small"

    # Prefijo de instrucción para T5/mT5 
    use_instruction_prefix: bool = True
    translation_prefix: str = "translate Chinese to English: "

    # Entrenamiento
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 3
    fraction: float = 1

cfg = Config()
print(cfg)


Config(dataset_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/processed_data/wuxia_zh_en_clean'), output_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/models/mt5_small'), ckpt_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/checkpoints'), training_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/training'), src_col='zh', tgt_col='en', model_ckpt='google/mt5-small', use_instruction_prefix=True, translation_prefix='translate Chinese to English: ', seed=42, max_source_length=128, max_target_length=128, batch_size=16, epochs=10, learning_rate=2e-05, weight_decay=0.01, early_stopping_patience=3, fraction=1)


In [4]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)



# Semillas para 
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # Para reproducibilidad estricta 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas y backend configurado.")


Usando dispositivo: cuda
Semillas fijadas y backend configurado.


## 3) Cargar dataset (Hugging Face Datasets)

In [5]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"No se encuentra el dataset en: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columnas
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' debe contener columnas '{src_col}' y '{tgt_col}'. Columnas: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

# Submuestreo 
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fracción={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


DatasetDict({
    train: Dataset({
        features: ['zh', 'en'],
        num_rows: 417208
    })
    validation: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
    test: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
})
{'zh': ['第章 听到白小纯的话语，看到圣皇的迟疑，邪皇这里顿时呼吸一促，他目中刹那就露出凌厉之芒，右手猛的一挥，顿时那残破的红日，骤然幻化', '尤其是看到人群内的宋缺时，神算子立刻警惕，他当年在空城，是第一个跟随白小纯的，受到了重用，如今却成为了第二个，他顿时就视宋缺为竞争对手'], 'en': ['chapter- The Saint-Emperor hesitated, and the Vile-Emperor sucked in a breath, eyes flickering with cold light as he waved his hand to summon his damaged red sun', 'That was even more the case when he noticed Song Que among Bai Xiaochun’s men, which immediately got him even more on guard. Back in Sky City, Master God-Diviner had been the first to start following Bai Xiaochun again, and it had led to incredible benefits. Now, he was only the second to join him, which put Song Que in his sights as a major rival']}
Tam. train/val/test (fracción=1): 417208 52151 52151

## 4) Tokenizer

In [6]:

from transformers import MT5Tokenizer, MT5ForConditionalGeneration

# Tokenizer
tokenizer = MT5Tokenizer.from_pretrained(cfg.model_ckpt)

# Modelo
model = MT5ForConditionalGeneration.from_pretrained(cfg.model_ckpt)

#  tokens especiales <id_0> etc
# Pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Decoder start token
if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id

# EOS token
if model.config.eos_token_id is None:
    model.config.eos_token_id = tokenizer.eos_token_id

model.config.use_cache = False

# Enviar a dispositivo
model.to(device)

# Info
print("Modelo:", cfg.model_ckpt)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros totales: {n_params:,}")
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)
print("decoder_start_token_id:", model.config.decoder_start_token_id)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.
You are using the default legacy behaviour of the <class 'transformers.models.mt5.tokenization_mt5.MT5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Modelo: google/mt5-small
Parámetros totales: 300,176,768
pad_token_id: 0
eos_token_id: 1
decoder_start_token_id: 0


## 5) Preprocesamiento

In [ ]:

max_source_length = cfg.max_source_length
max_target_length = cfg.max_target_length

def preprocess_function(examples):
    # Source (ZH) con prefijo de instrucción para mT5
    if cfg.use_instruction_prefix:
        src_texts = [cfg.translation_prefix + s.strip() for s in examples[cfg.src_col]]
    else:
        src_texts = [s.strip() for s in examples[cfg.src_col]]

    # Target (EN)
    tgt_texts = [t.strip() for t in examples[cfg.tgt_col]]

    # 3) Tokenización: text_target=
    model_inputs = tokenizer(
        src_texts,
        max_length=max_source_length,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=tgt_texts,           # importante para que cree labels bien
        max_length=max_target_length,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed) if "test" in tokenized_datasets else val_ds

print("Ejemplo tokenizado:", {k: type(v) for k,v in train_ds[0].items()})


Ejemplo tokenizado: {'input_ids': <class 'list'>, 'attention_mask': <class 'list'>, 'labels': <class 'list'>}


## 6) Data collator

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    pad_to_multiple_of=8,   # GPU
    return_tensors="pt"
)

# Ejemplo de batch
batch = data_collator([train_ds[i] for i in range(2)])
for k, v in batch.items():
    print(f"{k}: shape={v.shape}, dtype={v.dtype}")


input_ids: shape=torch.Size([2, 64]), dtype=torch.int64
attention_mask: shape=torch.Size([2, 64]), dtype=torch.int64
labels: shape=torch.Size([2, 96]), dtype=torch.int64
decoder_input_ids: shape=torch.Size([2, 96]), dtype=torch.int64


## 7) Configuración del entrenamiento


> **Por defecto**          
> **Optimizador**: `AdamW` (con LR=2e-5, weight decay=0.01) de `Seq2SeqTrainer`   
> **Loss**: `CrossEntropyLoss` (token-level) de `AutoModelForSeq2SeqLM `





In [ ]:
from transformers import Seq2SeqTrainer

class MySeq2SeqTrainer(Seq2SeqTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Sobrescribe compute_loss para asegurar que siempre se usan los labels. Evita <id_0>
        Ignora argumentos extra (ej. num_items_in_batch) que el modelo no acepta.
        """
        labels = inputs.get("labels")

        # forward solo con los argumentos que mT5 acepta
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            labels=labels,
        )

        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss


In [ ]:

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Carpeta para resultados
run_dir = cfg.output_dir
run_dir.mkdir(parents=True, exist_ok=True)

# Argumentos de entrenamiento 
training_args = Seq2SeqTrainingArguments(
    output_dir=str(run_dir),
    overwrite_output_dir=True,
    eval_strategy="epoch",          
    save_strategy="epoch",
    save_total_limit=3,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    num_train_epochs=cfg.epochs,
    weight_decay=cfg.weight_decay,
    logging_steps=200,
    predict_with_generate=True,           # <- importante para  traducción
    generation_max_length=cfg.max_target_length,
    generation_num_beams=4,
    fp16=False, #torch.cuda.is_available(), En este caso de mt5 NO se usa fp16 por razones de velocidad/optimizacion
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_grad_norm=1.0,
)

# Trainer para Seq2Seq
trainer = MySeq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator
)


print(" Seq2SeqTrainer configurado (PyTorch) con mT5.")


 Seq2SeqTrainer configurado (PyTorch) con mT5.


C:\Users\Usuario\AppData\Local\Temp\ipykernel_14892\3092238726.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `MySeq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = MySeq2SeqTrainer(


## 8) Entrenamiento

In [ ]:

from transformers import EarlyStoppingCallback
import json

# Añadir callback de early stopping
trainer.add_callback(EarlyStoppingCallback(
    early_stopping_patience=cfg.early_stopping_patience,  # nº de evaluaciones sin mejora
    early_stopping_threshold=0.0
))

# Entrenar
train_result = trainer.train()

# Guardar modelo final y tokenizer
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

# Guardar métricas de entrenamiento
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

# Guardar configuración y resultados en JSON 
run_info = {
    "model_name": cfg.model_ckpt,
    "epochs": cfg.epochs,
    "batch_size": cfg.batch_size,
    "learning_rate": cfg.learning_rate,
    "weight_decay": cfg.weight_decay,
    "train_size": len(train_ds),
    "val_size": len(val_ds),
    "metrics": metrics
}

with open(cfg.output_dir / "run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=4, ensure_ascii=False)

print(" Entrenamiento finalizado, modelo y artefactos guardados en:", cfg.output_dir)


Epoch,Training Loss,Validation Loss
1,3.121900,2.540218
2,2.767300,2.291648
3,2.616200,2.166946
4,2.491300,2.088592
5,2.418200,2.035139
6,2.390900,1.996091
7,2.341100,1.968712
8,2.304400,1.952548
9,2.317300,1.941656
10,2.285300,1.937261


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


***** train metrics *****
  epoch                    =                10.0
  total_flos               =         277496783GF
  train_loss               =              2.5973
  train_runtime            = 3 days, 22:20:40.29
  train_samples_per_second =              12.284
  train_steps_per_second   =               0.768
 Entrenamiento finalizado, modelo y artefactos guardados en: c:\Users\Usuario\Desktop\TFG\CORPUS\models\pretrain_mt5_small
